# 08 · Geospatial data and maps

**Session 8 — Space and Time** · about 45 minutes

**You will learn to**
- read boundaries into a GeoDataFrame and check its coordinate reference system (CRS)
- reproject before measuring areas or distances
- join tables to shapes **by code** and map rates, not counts
- draw honest choropleths: colour scale, legend, scale bar and north arrow

**Data:** comarques and municipalities (ICGC), Barcelona neighbourhoods (Open Data BCN), BCN500 Spain (CNIG), electricity consumption (Transparència Catalunya), traffic accidents 2024 (Open Data BCN).

**Outline**
0. Before you start (no AI)
1. GeoDataFrames and CRS
2. Areas need a projected CRS
3. Join by code, map a rate
4. Colour scales
5. Barcelona neighbourhoods
6. Spain with insets
7. Bivariate maps (optional)
8. Your map

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

In [ ]:
comarques = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-comarques-250000-20250730.json")

# Synthetic data, only for this exercise
rng = np.random.default_rng(11)
density = rng.lognormal(0, 0.6, len(comarques))
density[comarques["NOMCOMAR"].eq("Barcelonès").to_numpy()] *= 6
toy = comarques.assign(density=density, count=density * comarques["AREAC5000"] / 20)

fig, axs = plt.subplots(2, 2, figsize=(9, 8), layout="constrained")
toy.plot(column="count", cmap="viridis", legend=True, ax=axs[0, 0])
toy.plot(column="density", cmap="viridis", legend=True, ax=axs[0, 1])
toy.plot(column="density", cmap="jet", legend=True, ax=axs[1, 0])
toy.plot(column="density", cmap="viridis", ax=axs[1, 1])
axs[1, 1].set_aspect(0.45)
for ax, letter in zip(axs.flat, "ABCD"):
    ax.set_title(letter, loc="left", fontweight="bold")
    ax.set_axis_off()
fig.suptitle("Defibrillators by comarca (synthetic data)")
plt.show()

On paper: **which map answers “where is coverage highest relative to the size of the comarca”?**
What is wrong with each of the other three? (Answers at the end of section 4.)

## 1. GeoDataFrames and CRS

A GeoDataFrame is a DataFrame plus a **geometry** column (points, lines, polygons) and a **CRS**
(coordinate reference system) that says what the coordinates mean.

- `EPSG:4326`: longitude/latitude in degrees (GPS, web maps).
- `EPSG:25831`: ETRS89 / UTM zone 31N, in **metres** — the official projection in Catalonia.

Boundary files: GeoJSON (one file) or shapefile (`.shp` + `.shx` + `.dbf` + `.prj`, keep them together).

In [ ]:
print(comarques.crs)
comarques.head(3)

## 2. Areas need a projected CRS

### Verify the AI

An assistant computed comarca areas like this. Run it: what are the units?

In [ ]:
comarques.geometry.area.head(3)    # GeoPandas even warns you

Degrees² are not an area. Reproject to metres first, then compare with the official area (`AREAC5000`, km²):

In [ ]:
comarques_m = comarques.to_crs(25831)
comarques_m["area_km2"] = comarques_m.area / 1e6
comarques_m[["NOMCOMAR", "area_km2", "AREAC5000"]].head()

## 3. Join by code, map a rate

Electricity consumption per municipality → per comarca (via the municipalities table) → per km².
The keys need preparing (see notebook 02): INE's 5-digit code on one side, ICGC's 6-digit code on the other.

In [ ]:
energy = pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet").query("any == 2023")
municipis = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-municipis-1000000-20250730.json")
energy["ine5"] = energy["cdmun"].astype(str).str.zfill(5)
municipis["ine5"] = municipis["CODIMUNI"].str[:5]

step = energy.merge(municipis[["ine5", "CODICOMAR"]], on="ine5", how="left", validate="many_to_one")
print("rows before/after:", len(energy), len(step), "| unmatched:", step["CODICOMAR"].isna().sum())

per_comarca = step.groupby("CODICOMAR", as_index=False).agg(gwh=("consum_kwh", lambda s: s.sum() / 1e6))
cat = comarques_m.merge(per_comarca, on="CODICOMAR", how="left", validate="one_to_one")
cat["gwh_per_km2"] = cat["gwh"] / cat["area_km2"]
print("comarques without data:", cat["gwh"].isna().sum())

A map needs a **legend**, a **scale** and an **orientation**. Two small helpers (projected CRS in metres):

In [ ]:
def add_scale_bar(ax, length_km=50, loc=(0.05, 0.05)):
    """Draw a scale bar of length_km on an Axes whose coordinates are in metres."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = x0 + loc[0] * (x1 - x0)
    y = y0 + loc[1] * (y1 - y0)
    ax.plot([x, x + length_km * 1000], [y, y], color="black", lw=3)
    ax.text(x + length_km * 500, y + 0.015 * (y1 - y0), f"{length_km} km", ha="center", va="bottom", fontsize=9)


def add_north_arrow(ax, loc=(0.93, 0.88)):
    ax.annotate("N", xy=loc, xytext=(loc[0], loc[1] - 0.1), xycoords="axes fraction",
                ha="center", va="center", fontsize=12, fontweight="bold",
                arrowprops={"arrowstyle": "-|>", "color": "black"})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cat.plot(column="gwh", cmap="viridis", legend=True, ax=axs[0],
         legend_kwds={"label": "GWh (2023)", "shrink": 0.7})
axs[0].set_title("Count: total consumption")
cat.plot(column="gwh_per_km2", cmap="viridis", legend=True, ax=axs[1],
         legend_kwds={"label": "GWh per km² (2023)", "shrink": 0.7})
axs[1].set_title("Rate: consumption per km²")
for ax in axs:
    ax.set_axis_off()
    add_scale_bar(ax)
    add_north_arrow(ax)
plt.show()

## 4. Colour scales

In the rate map, the Barcelonès makes every other comarca the same dark colour. A **log scale** or
**quantile classes** show the variation among the rest. Whatever you choose, say it in the legend.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cat.plot(column="gwh_per_km2", cmap="viridis", norm=LogNorm(), legend=True, ax=axs[0],
         legend_kwds={"label": "GWh per km² (log scale)", "shrink": 0.7})
axs[0].set_title("Log scale")

cat["quintile"] = pd.qcut(cat["gwh_per_km2"], 5, labels=["lowest 20%", "2", "3", "4", "highest 20%"])
cat.plot(column="quintile", cmap="viridis", categorical=True, legend=True, ax=axs[1],
         legend_kwds={"title": "Quintile", "loc": "lower right", "fontsize": 8})
axs[1].set_title("Quantile classes")
for ax in axs:
    ax.set_axis_off()
    add_scale_bar(ax)
plt.show()

**Answers to section 0.** **B** answers the question (density, perceptually uniform colours, legend).
**A** maps counts, so large comarques dominate: it is mostly a map of area.
**C** uses a rainbow (`jet`): it invents boundaries between colours and is not readable in grey or by
people with colour-vision deficiency. **D** has no legend and is stretched: unprojected degrees with
a wrong aspect ratio.

## 5. Barcelona neighbourhoods

People involved in traffic accidents in 2024, by neighbourhood (*barri*). One accident can involve
several people: count **accident files** (`Numero_expedient`).

### TASK I — accidents per km² by neighbourhood

1. Read `bcn/barris.geojson` into `barris` (EPSG:25831) and compute `area_km2`.
2. Count distinct accidents per neighbourhood code in `accidents` (watch the key: `"57"` vs `"01"`).
3. Build `bcn` = `barris` joined with the counts (`validate="one_to_one"`), with `accidents` and
   `per_km2` (0 where a neighbourhood has no accident).
4. Map `per_km2` with a legend, a scale bar (2 km) and a north arrow.

In [ ]:
accidents = pd.read_parquet(DATA / "examples/bcn_accidents_2024.parquet")
accidents[["Numero_expedient", "Codi_barri", "Nom_barri"]].head()

In [ ]:
# TASK I — your code here
bcn = None

In [ ]:
# ✅ Self-check TASK I
assert bcn is not None and len(bcn) == 73, "All 73 neighbourhoods should stay in the map."
assert bcn.crs.is_projected, "Work in a projected CRS (metres)."
assert bcn["per_km2"].notna().all()
assert bcn["accidents"].sum() == accidents.loc[accidents["Codi_barri"].str.zfill(2).isin(bcn["codi_barri"]),
                                                "Numero_expedient"].nunique()
print("✅ TASK I passed")

## 6. Spain with insets

BCN500 boundaries (CNIG). The Canary and Balearic Islands go into insets so the peninsula stays large.

In [ ]:
spain = gpd.read_file(DATA / "carto/BCN500_0101S_LIMITE_ADM.shp")
spain = spain[~spain["CCAA"].isin(["SAHARA OCCIDENTAL", "ALGERIA", "PORTUGAL", "MARRUECOS",
                                   "MAURITANIA", "FRANCIA", "ANDORRA"])]
mainland = spain[~spain["CCAA"].isin(["Illes Balears", "Canarias"])]

fig, ax = plt.subplots(figsize=(8, 8))
mainland.plot(ax=ax, color="lightgrey", edgecolor="white", lw=0.3)
canarias = ax.inset_axes([0.0, 0.0, 0.3, 0.25])
spain[spain["CCAA"] == "Canarias"].plot(ax=canarias, color="lightgrey")
balears = ax.inset_axes([0.75, 0.35, 0.25, 0.25])
spain[spain["CCAA"] == "Illes Balears"].plot(ax=balears, color="lightgrey")
for a in (ax, canarias, balears):
    a.set_xticks([])
    a.set_yticks([])
plt.show()

## 7. Bivariate maps (optional)

The session opener showed a bivariate map: two variables, each cut into classes, combined into one
colour. Here: consumption per km² × share of industrial consumption, 3 × 3 classes.

In [ ]:
industrial = (step[step["descripcio_sector"] == "INDUSTRIAL"].groupby("CODICOMAR")["consum_kwh"].sum()
              / step.groupby("CODICOMAR")["consum_kwh"].sum()).rename("industrial_share")
biv = cat.merge(industrial.reset_index(), on="CODICOMAR", validate="one_to_one")
biv["cx"] = pd.qcut(biv["gwh_per_km2"], 3, labels=False)
biv["cy"] = pd.qcut(biv["industrial_share"], 3, labels=False)

palette = [["#e8e8e8", "#ace4e4", "#5ac8c8"],      # rows: industrial share low → high
           ["#dfb0d6", "#a5add3", "#5698b9"],      # columns: density low → high
           ["#be64ac", "#8c62aa", "#3b4994"]]
biv["colour"] = [palette[y][x] for x, y in zip(biv["cx"], biv["cy"])]

fig, ax = plt.subplots(figsize=(7, 7))
biv.plot(color=biv["colour"], edgecolor="white", lw=0.4, ax=ax)
ax.set_axis_off()
legend = ax.inset_axes([0.0, 0.0, 0.22, 0.22])
legend.imshow(np.array([[[int(c[i:i + 2], 16) for i in (1, 3, 5)] for c in row] for row in palette]),
              origin="lower")
legend.set_xticks([0, 2], ["low", "high"], fontsize=7)
legend.set_yticks([0, 2], ["low", "high"], fontsize=7)
legend.set_xlabel("GWh/km² →", fontsize=7)
legend.set_ylabel("industrial share →", fontsize=7)
plt.show()

Is this faster to read than two separate maps? (The S8 opener debate.)

## 8. Your map

For your project: join by a **code**, print the row counts before and after, map a **rate**, choose
the colour scale on purpose and add a legend, a scale bar and a north arrow.

**Next:** `09_time_series.ipynb`.